# YOLO26n detection and metric depth parking screen

Revision `PARKING-EDGE-SCREEN-2026-10-21-r1` — latest 21 October 2026. Run top to bottom on a **CPU runtime**. No GPU is required. No training, KD, depth-scale fitting, Core ML, historical runtime restore or phone installation.

Runs YOLO26n at 640 and YOLO26n-depth at 768. Estimates nominal camera-axis depth from the central half of each detection box. This ROI is not an instance mask or the nearest surface. Released depth scale is not independently validated on your camera.

The fixed 64-frame sample deliberately includes near-range examples. It is **not full KITTI AP**. Detection recall bins use annotated box-center depth only to identify nearby objects. Numeric depth is never scored against those box centers. Independent surface-distance accuracy and iPhone performance remain unmeasured.

## 1 Mount Drive and update main

Both candidate notebooks share the same input cache but use different pinned Ultralytics versions. Start a fresh CPU session when switching notebooks. Keep completed output folders; change RUN_ID if source, environment or model identity changes. Official checkpoints contain Python objects: only the pinned hash-verified upstream files are loaded.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, subprocess, sys
from google.colab import drive, files
drive.mount('/content/drive')
REVISION = 'PARKING-EDGE-SCREEN-2026-10-21-r1'  # Latest 21 October 2026
CANDIDATE = 'yolo_depth'
RUN_ID = 'yolo_depth_screen_r1'
PROJECT = Path('/content/mobile_adas3d')
URL = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',URL,str(PROJECT)],check=True)
else:
    branch = subprocess.check_output(['git','-C',str(PROJECT),'branch','--show-current'],text=True).strip()
    if branch != 'main': raise RuntimeError(f'Expected main, found {branch}; preserve the checkout')
    for command in (['diff','--quiet'],['diff','--cached','--quiet']):
        if subprocess.run(['git','-C',str(PROJECT),*command]).returncode:
            raise RuntimeError('Preserve tracked changes before updating')
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only'],check=True)
BASE = Path('/content/drive/MyDrive/mobile_adas3d_outputs')
OUTPUT = BASE/'parking_edge_candidates'/RUN_ID
CACHE = BASE/'input_caches/parking_edge_fixed64_v1'
DATA = Path('/content/kitti_parking_edge_fixed64_v1')
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
SPLIT = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen/val.txt')
WEIGHTS = BASE/'parking_edge_candidates/pinned_checkpoints_v1'
SOURCE = Path('/content/UrbanOmniDetect2_PARKING_r1')
SCRIPT = PROJECT/'scripts/screen_parking_edge_candidates.py'
CONFIG = PROJECT/'configs/parking_edge_candidates_v1.json'
for p in (SCRIPT, CONFIG):
    if not p.is_file(): raise FileNotFoundError(f'{p} missing; use the published project main')
if json.loads(CONFIG.read_text())['revision'] != REVISION:
    raise RuntimeError('Notebook and project revision differ; reload the matching notebook')
def run_logged(command, name, allow_failure=False):
    command = [str(v) for v in command]
    logs = OUTPUT/'colab_logs'; logs.mkdir(parents=True,exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = logs/f'{name}_{stamp}.log'
    print('+',shlex.join(command),'\nDurable log:',path,flush=True)
    tail = deque(maxlen=30)
    with path.open('w') as log:
        with subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
            status=process.wait()
        log.write(f'\nEXIT CODE: {status}\n')
    if status and not allow_failure:
        raise RuntimeError(f'Exit {status}; log={path}\n'+'\n'.join(tail))
    return status
print(REVISION, CANDIDATE, '\nOutput:',OUTPUT)
print('Project commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip())


## 2 Stage 64 frames with four copy workers

This step uses only the standard library. The first run indexes the small validation labels and copies only 64 images and labels. Subsequent sessions restore one hash-verified archive instead of repeating thousands of Drive reads. It does not select frames based on model predictions.

In [ ]:
run_logged([sys.executable,'-u',PROJECT/'scripts/parking_edge_common.py',
            '--dataset-root',DATASET_ROOT,'--split',SPLIT,'--cache',CACHE,
            '--local-data',DATA,'--workers','4'],'stage_fixed64')
inputs=json.loads((DATA/'input_manifest.json').read_text())
assert len(inputs['ids'])==64 and not inputs['gpu_used']
print('Fixed input identity:',inputs['manifest_sha256'])
print('Available frames by class and true box-center band:',inputs['available_frames_per_cell'])


## 3 Install this candidate runtime and run tests

This is a new CPU screen, not a restoration of an old training environment. Existing PyTorch is recorded rather than replaced. The package version is pinned for this candidate. All model execution happens in fresh subprocesses so notebook imports cannot shadow the model code. Do not run the other candidate notebook in this same session.

In [ ]:
run_logged([sys.executable,'-m','pip','install','-q',
            'ultralytics==8.4.183','onnx==1.20.1','onnxruntime==1.30.0'],'candidate_dependencies')
run_logged([sys.executable,'-m','unittest','discover','-s','tests',
            '-p','test_parking_edge_candidates.py','-v'],'candidate_tests')


## 4 Export and compare PyTorch with ONNX CPU

This downloads pinned official checkpoints, exports fixed FP32 ONNX graphs and checks standard operators. It then compares detections on all 64 frames and numeric original-resolution depth maps. It saves 16 overlays and per-frame results. Resume verifies completed frame identities. Two warm-up frames are excluded from pipeline timing; raw ONNX timing uses two CPU threads and 20 measured repetitions. Neither measurement is phone or live-camera performance.

A failure is preserved in a JSON report and the durable log. Continue to the final download cell even if export fails; return that diagnostic bundle rather than changing tolerances.

In [ ]:
SCREEN_CODE=run_logged([sys.executable,'-X','faulthandler','-u',SCRIPT,'screen',
                       '--candidate',CANDIDATE,'--output',OUTPUT,'--local-data',DATA,
                       '--weights',WEIGHTS,'--source',SOURCE],'candidate_screen',allow_failure=True)
REPORT=OUTPUT/'candidate_screen.json'
if REPORT.is_file():
    report=json.loads(REPORT.read_text())
    print(json.dumps({k:report[k] for k in ('candidate','complete','technical_export_screen_passed',
                     'frames','metric_range_available','independent_distance_accuracy_status',
                     'iphone_performance_measured','parking_validation_qualified')},indent=2))
    print('ONNX neural-only warm CPU timing:',report['raw_onnx_profile']['sequential_pipeline_neural_only'])
else:
    print('No completed screen report. Return the diagnostic ZIP from section 6; inspect the screen log.')
if SCREEN_CODE:
    print('STOP before any deployment. Diagnostics will still be bundled below.')


## 5 Inspect sample overlays

Green is Vehicle and yellow is Pedestrian. Distance text is an unvalidated central-box surface-depth estimate. Large objects, overlapping objects and background inside the box can bias it.

In [ ]:
from IPython.display import display, Image
paths=sorted((OUTPUT/'overlays').glob('*.png'))
for p in paths[:4]:
    print(p.name);display(Image(filename=str(p)))
if not paths: print('No overlays were produced; return the failure bundle.')


## 6 Download results

Return this ZIP for review. It includes ONNX graphs, fixed input/output tensor fixtures, overlays, per-frame predictions, timings, provenance and failure logs. It excludes pickle checkpoints and the KITTI source dataset. A technical export pass does not authorize sensor validation or claim an iPhone speed target.

Next decision: compare measured CPU cost and detection quality first, then test a selected package on iPhone and assess distance against independently referenced mounted-camera measurements. No KD or new training starts automatically.

In [ ]:
run_logged([sys.executable,'-u',SCRIPT,'bundle','--candidate',CANDIDATE,'--output',OUTPUT],'bundle')
bundles=sorted(OUTPUT.glob('*parking_edge_results*.zip'),key=lambda p:p.stat().st_mtime_ns)
if not bundles: raise FileNotFoundError('No result bundle; inspect the bundle log')
BUNDLE=bundles[-1]
print('Return:',BUNDLE)
files.download(str(BUNDLE))
